# Dr. Splat: Directly Referring 3D Gaussian Splatting
**CVPR 2025 Highlight**

1. 환경 설정
2. 데이터셋 다운로드 (LeRF-OVS)
3. 기본 3DGS 학습 (30K iterations)
4. Feature 추출 (SAM + OpenCLIP)
5. Dr. Splat Feature Registration
6. 시각화 (PCA / Activation)

---
## 0. Configuration

In [ ]:
import os, shutil, glob

WORK_DIR = "/content/drsplat"
MY_GITHUB_REPO = "https://github.com/BAEJUNHAK/Dr-Splat.git"

SCENE_NAME = "figurines"  # figurines / ramen / teatime / waldo_kitchen

DATASET_ROOT = f"{WORK_DIR}/data/lerf_ovs"
SCENE_PATH = f"{DATASET_ROOT}/{SCENE_NAME}"

DRSPLAT_REPO = f"{WORK_DIR}/Dr-Splat"
GS_REPO = f"{WORK_DIR}/gaussian-splatting"

GS_OUTPUT_PATH = f"{WORK_DIR}/output/3dgs_{SCENE_NAME}"
GS_CHECKPOINT = f"{GS_OUTPUT_PATH}/chkpnt30000.pth"
DRSPLAT_OUTPUT_PATH = f"{WORK_DIR}/output/drsplat_{SCENE_NAME}"

# Google Drive 백업 경로
DRIVE_SAVE_DIR = "/content/drive/MyDrive/drsplat_results"

GPU_ID = 0
TOPK = 40

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Scene: {SCENE_NAME}")
print(f"Scene path: {SCENE_PATH}")

---
## 1. GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 2. 환경 설정

### 2-1. 레포 클론

In [ ]:
# Dr-Splat (본인 GitHub) - .git 없으면 삭제 후 재클론
if os.path.exists(DRSPLAT_REPO) and os.path.isdir(os.path.join(DRSPLAT_REPO, '.git')):
    print("Dr-Splat exists. Pulling latest...")
    !cd {DRSPLAT_REPO} && git pull origin main
else:
    !rm -rf {DRSPLAT_REPO}
    !cd {WORK_DIR} && git clone {MY_GITHUB_REPO} --recursive
    print("Dr-Splat cloned!")

# 서브모듈 초기화 (중요!)
!cd {DRSPLAT_REPO} && git submodule update --init --recursive
print("\n=== submodules ===")
!ls {DRSPLAT_REPO}/submodules/

In [ ]:
# 원본 3DGS (기본 학습용)
if not os.path.exists(GS_REPO):
    !cd {WORK_DIR} && git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive
    print("3DGS cloned!")
else:
    print("3DGS already exists.")

### 2-2. 패키지 설치
Colab 기본 PyTorch (CUDA 12.x)를 그대로 사용합니다.

In [ ]:
!pip install plyfile tqdm scipy scikit-image matplotlib opencv-python-headless -q
!pip install open-clip-torch faiss-cpu kmeans_pytorch -q
!pip install tensorboard tyro jaxtyping gdown -q

### 2-3. CUDA 서브모듈 빌드
> 빌드에 몇 분 걸릴 수 있습니다.

In [ ]:
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["CUDA_PATH"] = "/usr/local/cuda"

# 원본 3DGS 서브모듈 (기본 학습용)
!pip install {GS_REPO}/submodules/diff-gaussian-rasterization
!pip install {GS_REPO}/submodules/simple-knn

---
## 3. Google Drive 마운트 & 데이터셋 다운로드
체크포인트 자동 저장/복원을 위해 먼저 Drive를 마운트합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Drive backup dir: {DRIVE_SAVE_DIR}")

### 3-1. LeRF-OVS 다운로드

| 소스 | 링크 |
|------|------|
| LangSplat Google Drive | [링크](https://drive.google.com/file/d/1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt/view) |
| OpenGaussian Baidu | [링크](https://pan.baidu.com/s/1B_tGYla5dWyJRu3jTNTMvA?pwd=u5iy) |

방법 1부터 시도하세요.

#### 방법 1: gdown (권장)

In [ ]:
import gdown, shutil

DOWNLOAD_DIR = f"{WORK_DIR}/data"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

GDRIVE_ID = "1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt"
ZIP_PATH = f"{DOWNLOAD_DIR}/lerf_ovs.zip"

if not os.path.exists(SCENE_PATH):
    print("Downloading LeRF-OVS ...")
    try:
        gdown.download(id=GDRIVE_ID, output=ZIP_PATH, quiet=False, fuzzy=True)
        if os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 1_000_000:
            print(f"Download: {os.path.getsize(ZIP_PATH) / 1e9:.2f} GB")
            !cd {DOWNLOAD_DIR} && unzip -q -o {ZIP_PATH}
            !rm -f {ZIP_PATH}

            # 중첩 폴더 자동 처리: data/lerf_ovs/lerf_ovs/ -> data/lerf_ovs/
            nested = os.path.join(DATASET_ROOT, "lerf_ovs")
            if os.path.isdir(nested) and not os.path.isdir(os.path.join(DATASET_ROOT, SCENE_NAME)):
                tmp = f"{DOWNLOAD_DIR}/_lerf_tmp"
                shutil.move(nested, tmp)
                shutil.rmtree(DATASET_ROOT)
                shutil.move(tmp, DATASET_ROOT)
                print("Nested folder fixed!")
            print("Done!")
        else:
            print("[WARN] gdown 실패. 방법 2를 사용하세요.")
            !rm -f {ZIP_PATH}
    except Exception as e:
        print(f"[ERROR] {e}\n방법 2를 사용하세요.")
else:
    print(f"Dataset exists: {SCENE_PATH}")

#### 방법 2: Google Drive 마운트 (gdown 실패시)

In [ ]:
# === 주석 해제 후 실행 ===

# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ZIP = "/content/drive/MyDrive/lerf_ovs.zip"  # <-- 경로 수정
# !cp {DRIVE_ZIP} {DOWNLOAD_DIR}/
# !cd {DOWNLOAD_DIR} && unzip -q -o lerf_ovs.zip && rm lerf_ovs.zip

### 3-2. 데이터 구조 검증

In [ ]:
print("=== Available scenes ===")
if os.path.exists(DATASET_ROOT):
    for d in sorted(os.listdir(DATASET_ROOT)):
        full = os.path.join(DATASET_ROOT, d)
        if os.path.isdir(full):
            has_img = os.path.isdir(os.path.join(full, 'images'))
            has_sp = os.path.isdir(os.path.join(full, 'sparse', '0'))
            n = len(os.listdir(os.path.join(full, 'images'))) if has_img else 0
            print(f"  {d:20s} images:{n:3d}  sparse:{'OK' if has_sp else 'X'}")
else:
    print(f"[ERROR] {DATASET_ROOT} not found!")

def verify(path, name):
    checks = {
        'images/': os.path.isdir(os.path.join(path, 'images')),
        'sparse/0/cameras.bin': os.path.exists(os.path.join(path, 'sparse', '0', 'cameras.bin')),
        'sparse/0/images.bin': os.path.exists(os.path.join(path, 'sparse', '0', 'images.bin')),
        'sparse/0/points3D.bin': os.path.exists(os.path.join(path, 'sparse', '0', 'points3D.bin')),
    }
    ok = all(checks.values())
    print(f"\n{'PASS' if ok else 'FAIL'} - {name}")
    for k, v in checks.items():
        print(f"  {'[O]' if v else '[X]'} {k}")

verify(SCENE_PATH, SCENE_NAME)

### 3-3. SAM 체크포인트 다운로드

In [ ]:
SAM_CKPT = f"{DRSPLAT_REPO}/ckpts/sam_vit_h_4b8939.pth"
os.makedirs(f"{DRSPLAT_REPO}/ckpts", exist_ok=True)

if not os.path.exists(SAM_CKPT):
    print("Downloading SAM ViT-H...")
    !wget -q -P {DRSPLAT_REPO}/ckpts/ https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    print("Done!")
else:
    print("SAM checkpoint exists.")
!ls -lh {SAM_CKPT}

---
## 4. 기본 3DGS 학습 (30K iterations)
> T4 기준 약 20~40분 소요

In [ ]:
os.makedirs(GS_OUTPUT_PATH, exist_ok=True)

# Drive에서 체크포인트 확인
drive_gs_dir = os.path.join(DRIVE_SAVE_DIR, f"3dgs_{SCENE_NAME}")
drive_ckpt = os.path.join(drive_gs_dir, "chkpnt30000.pth")

if os.path.exists(GS_CHECKPOINT):
    # 로컬에 이미 있음
    print(f"Checkpoint exists (local): {GS_CHECKPOINT}")

elif os.path.exists(drive_ckpt):
    # Drive에서 복원
    print(f"Restoring checkpoint from Drive: {drive_gs_dir}")
    shutil.copytree(drive_gs_dir, GS_OUTPUT_PATH, dirs_exist_ok=True)
    print(f"Restored! {GS_CHECKPOINT}")

else:
    # 학습 실행
    print("Starting 3DGS training (30K iterations)...")
    !cd {GS_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python train.py \
        -s {SCENE_PATH} \
        -m {GS_OUTPUT_PATH} \
        --iterations 30000 \
        --save_iterations 30000 \
        --checkpoint_iterations 30000 \
        --eval
    print("Training done!")

    # 학습 완료 후 Drive에 자동 저장
    if os.path.exists(GS_CHECKPOINT):
        print(f"\nSaving checkpoint to Drive: {drive_gs_dir}")
        shutil.copytree(GS_OUTPUT_PATH, drive_gs_dir, dirs_exist_ok=True)
        print("Saved!")

In [ ]:
!ls -lh {GS_OUTPUT_PATH}/chkpnt*.pth 2>/dev/null || echo "No checkpoint found!"

---
## 5. Feature 추출 (SAM + OpenCLIP)
> **중요**: langsplat-rasterization으로 교체합니다.

In [ ]:
# 서브모듈 확인
!cd {DRSPLAT_REPO} && git submodule update --init --recursive
!ls {DRSPLAT_REPO}/submodules/

# 원본 rasterizer 제거 -> langsplat 버전 설치
!pip uninstall diff-gaussian-rasterization -y -q 2>/dev/null
!pip install {DRSPLAT_REPO}/submodules/langsplat-rasterization
!pip install {DRSPLAT_REPO}/submodules/segment-anything-langsplat

In [ ]:
# CLIP 레포
CLIP_DIR = f"{DRSPLAT_REPO}/CLIP"
if not os.path.exists(CLIP_DIR):
    !cd {DRSPLAT_REPO} && git clone https://github.com/openai/CLIP.git
else:
    print("CLIP exists.")

In [ ]:
LF_DIR = f"{SCENE_PATH}/language_features"

if os.path.exists(LF_DIR) and len(os.listdir(LF_DIR)) > 0:
    print(f"Language features exist: {LF_DIR}")
    !ls {LF_DIR} | head -10
else:
    os.makedirs(LF_DIR, exist_ok=True)
    print("Starting feature extraction...")
    !cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python preprocessing.py \
        --dataset_path {SCENE_PATH} \
        --sam_ckpt_path {SAM_CKPT}
    print("Done!")

In [ ]:
import numpy as np
lf_files = sorted(os.listdir(LF_DIR))
f_files = [f for f in lf_files if f.endswith('_f.npy')]
print(f"Total: {len(lf_files)} files, {len(f_files)} features")
if f_files:
    print(f"Shape: {np.load(os.path.join(LF_DIR, f_files[0])).shape}")

---
## 6. Dr. Splat Feature Registration
Majority Voting + PQ 압축 (gradient 최적화 아님)

In [ ]:
PQ_INDEX = f"{DRSPLAT_REPO}/ckpts/pq_index.faiss"
if os.path.exists(PQ_INDEX):
    print(f"PQ index: {PQ_INDEX}")
else:
    print("[ERROR] PQ index not found!")
    !ls -la {DRSPLAT_REPO}/ckpts/

In [ ]:
print("Starting Dr. Splat...")
!cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python train.py \
    -s {SCENE_PATH} \
    -m {DRSPLAT_OUTPUT_PATH} \
    --start_checkpoint {GS_CHECKPOINT} \
    --feature_level 1 \
    --name_extra pq_openclip \
    --use_pq \
    --pq_index {PQ_INDEX} \
    --port 55560 \
    --topk {TOPK} \
    --eval
print("Done!")

# Dr. Splat 결과 자동 Drive 저장
drsplat_dirs = glob.glob(f"{WORK_DIR}/output/drsplat_{SCENE_NAME}*")
for d in drsplat_dirs:
    dst = os.path.join(DRIVE_SAVE_DIR, os.path.basename(d))
    print(f"Saving to Drive: {dst}")
    shutil.copytree(d, dst, dirs_exist_ok=True)
print("Saved!")

In [ ]:
import glob
drsplat_dirs = glob.glob(f"{WORK_DIR}/output/drsplat_{SCENE_NAME}*")
if drsplat_dirs:
    DRSPLAT_TRAINED_PATH = sorted(drsplat_dirs)[-1]
    print(f"Using: {DRSPLAT_TRAINED_PATH}")
    !ls -la {DRSPLAT_TRAINED_PATH}/
else:
    DRSPLAT_TRAINED_PATH = DRSPLAT_OUTPUT_PATH
    print(f"[WARN] Using default: {DRSPLAT_TRAINED_PATH}")

---
## 7. 시각화

### 7-1. PCA Feature Visualization

In [ ]:
!cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python render_pca.py \
    -s {SCENE_PATH} \
    -m {DRSPLAT_TRAINED_PATH} \
    --pq_index {PQ_INDEX} \
    --feature_level 1 \
    -l language_features_dim3

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

pca_candidates = glob.glob(f"{DRSPLAT_TRAINED_PATH}/**/renders_pca", recursive=True)
if pca_candidates:
    pca_images = sorted(glob.glob(f"{pca_candidates[0]}/*.png"))[:8]
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, p in zip(axes.flat, pca_images):
        ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8); ax.axis('off')
    plt.suptitle('PCA Feature Visualization', fontsize=16); plt.tight_layout(); plt.show()
else:
    print("PCA output not found")

### 7-2. Activation (텍스트 쿼리)

In [ ]:
TEXT_QUERY = "figurine"
SAVE_LABEL = "figurine_test"
THRESHOLD = 0.5

!cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python render_activation.py \
    -s {SCENE_PATH} \
    -m {DRSPLAT_TRAINED_PATH} \
    --semantic_model sam \
    --feature_level 1 \
    --pq_index {PQ_INDEX} \
    --img_label "{TEXT_QUERY}" \
    --img_save_label "{SAVE_LABEL}" \
    --threshold {THRESHOLD} \
    -l language_features_dim3

In [ ]:
act_images = []
for c in glob.glob(f"{DRSPLAT_TRAINED_PATH}/**/*{SAVE_LABEL}*", recursive=True):
    if os.path.isdir(c):
        act_images.extend(sorted(glob.glob(f"{c}/*.png"))[:4])
if act_images:
    n = min(len(act_images), 8)
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, p in zip(axes.flat, act_images[:n]):
        ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8); ax.axis('off')
    plt.suptitle(f'Activation: "{TEXT_QUERY}"', fontsize=16); plt.tight_layout(); plt.show()
else:
    print("No activation images found.")

### 7-3. 다양한 쿼리 테스트

In [ ]:
QUERIES = ["table", "cup", "plate", "flower"]

for query in QUERIES:
    print(f"\n>>> {query}")
    !cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python render_activation.py \
        -s {SCENE_PATH} -m {DRSPLAT_TRAINED_PATH} \
        --semantic_model sam --feature_level 1 --pq_index {PQ_INDEX} \
        --img_label "{query}" --img_save_label "{query}_test" \
        --threshold 0.5 -l language_features_dim3 2>&1 | tail -3
print("\nDone!")

---
## 8. 체크포인트 자동 저장/복원 요약

| 시점 | 동작 |
|------|------|
| **3DGS 학습 전** | Drive에 체크포인트 있으면 자동 복원 |
| **3DGS 학습 후** | Drive에 자동 저장 |
| **Dr. Splat 후** | Drive에 자동 저장 |
| **런타임 재시작** | 셀 순서대로 실행하면 Drive에서 복원되어 학습 스킵 |

---
## Troubleshooting

| 문제 | 해결 |
|------|------|
| **OOM** | `TOPK`를 10~20으로 줄이기 |
| **submodules 없음** | `git submodule update --init --recursive` 실행 |
| **rasterizer 충돌** | 5단계에서 uninstall -> langsplat 설치 확인 |
| **PQ index 없음** | `ckpts/pq_index.faiss` 확인 |
| **런타임 재시작 후 데이터 없음** | Drive 백업에서 복원 또는 재다운로드 |